# 🎬 Supernan Hindi Dubbing Pipeline

**Zero-cost pipeline: English video → Hindi dubbed video with lip sync**

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

Pipeline stages:
1. Install dependencies
2. Clone VideoReTalking & GFPGAN
3. Upload & extract 15-second clip
4. Whisper transcription
5. IndicTrans2 Hindi translation
6. Coqui XTTS v2 voice cloning
7. Audio speed adjustment
8. VideoReTalking lip-sync
9. GFPGAN face enhancement
10. Download output

In [2]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import subprocess, torch
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else 'No NVIDIA GPU.')
except FileNotFoundError:
    print('No nvidia-smi (Mac/CPU mode).')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


No nvidia-smi (Mac/CPU mode).
Device: cpu


In [3]:
# ── Cell 2: Install System Deps ───────────────────────────────────────────────
!apt-get install -y ffmpeg wget unzip -q
print('✓ ffmpeg installed')

zsh(51048) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51048) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh:1: command not found: apt-get
✓ ffmpeg installed


In [4]:
# ── Cell 3: Install Python packages ──────────────────────────────────────────
# PyTorch with CUDA (Colab default, but ensuring compatibility)
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

# Core pipeline deps
!pip install -q openai-whisper TTS transformers sentencepiece sacremoses
!pip install -q git+https://github.com/VarunGumma/IndicTransTokenizer.git
!pip install -q pydub librosa soundfile deep-translator
!pip install -q basicsr facexlib gfpgan realesrgan
print("✓ All packages installed")


zsh(51049) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51049) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
python3.11(51049) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
Python(51049) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(51050) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51050) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
python3.11(51050) MallocStackLogging: could not tag MSL-re

In [5]:
# ── Cell 4: Clone Repos ───────────────────────────────────────────────────────
import os

# VideoReTalking
if not os.path.exists('VideoReTalking'):
    !git clone -q https://github.com/vinthony/video-retalking VideoReTalking
    %pip install -q -r VideoReTalking/requirements.txt
print('✓ VideoReTalking ready')

# GFPGAN
if not os.path.exists('GFPGAN'):
    !git clone -q https://github.com/TencentARC/GFPGAN GFPGAN
    %pip install -q basicsr facexlib gfpgan
    %cd GFPGAN
    !python setup.py develop -q
    %cd ..
print('✓ GFPGAN ready')

✓ VideoReTalking ready
✓ GFPGAN ready


In [6]:
# ── Cell 5: Download VideoReTalking Weights ──────────────────────────────────
import os
os.makedirs("models/VideoReTalking", exist_ok=True)
!wget -q https://github.com/vinthony/video-retalking/releases/download/v0.0.1/30_net_G.pth -O models/VideoReTalking/30_net_G.pth
!wget -q https://github.com/vinthony/video-retalking/releases/download/v0.0.1/BFM.zip -O models/VideoReTalking/BFM.zip
!unzip -qo models/VideoReTalking/BFM.zip -d models/VideoReTalking/
print("✓ VideoReTalking weights ready")


zsh(51107) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51107) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
wget(51107) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(51113) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51113) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
wget(51113) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(51237) MallocStackLogging: process is not in a debuggable envi

In [7]:
# ── Cell 6: Download GFPGAN Weights ──────────────────────────────────────────
os.makedirs('GFPGAN/experiments/pretrained_models', exist_ok=True)
gfpgan_weight = 'GFPGAN/experiments/pretrained_models/GFPGANv1.4.pth'
if not os.path.exists(gfpgan_weight):
    !wget -q --show-progress -O {gfpgan_weight} https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth
print('✓ GFPGAN weights ready')

✓ GFPGAN weights ready


In [8]:
# ── Cell 7: Download Input Video from Google Drive ──────────────────────────
import os, subprocess

# ── Paste your Google Drive share link here ───────────────────────────────────
GDRIVE_URL = 'https://drive.google.com/file/d/1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW/view?usp=sharing'

INPUT_VIDEO = 'supernan_training.mp4'

if not os.path.exists(INPUT_VIDEO):
    print('Downloading video from Google Drive...')
    # Install gdown if needed
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    gdown.download(url=GDRIVE_URL, output=INPUT_VIDEO, fuzzy=True)

if os.path.exists(INPUT_VIDEO):
    size_mb = os.path.getsize(INPUT_VIDEO) / 1e6
    print(f'✓ Video ready: {INPUT_VIDEO} ({size_mb:.1f} MB)')
else:
    print('⚠️  Download failed. If the file is restricted, share it as "Anyone with the link"')
    print('   Or place the video manually in:', os.getcwd())


✓ Video ready: supernan_training.mp4 (621.9 MB)


In [9]:
# ── Cell 8: Configuration ────────────────────────────────────────────────────
# ⬇️ Change these if you want a different segment
START_SEC = 30
END_SEC   = 50

os.makedirs('workspace', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('models', exist_ok=True)

print(f'Processing segment: {START_SEC}s – {END_SEC}s ({END_SEC - START_SEC}s clip)')

Processing segment: 30s – 50s (20s clip)


In [10]:
# ── Cell 9: Stage 1 — Extract Clip ───────────────────────────────────────────
CLIP = 'workspace/clip.mp4'
duration = END_SEC - START_SEC

!ffmpeg -y -ss {START_SEC} -i "{INPUT_VIDEO}" -t {duration} \
    -c:v libx264 -c:a aac -preset fast -crf 18 {CLIP} -loglevel warning

from IPython.display import Video
print('✓ Clip extracted')
Video(CLIP, width=640)

zsh(51241) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51241) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(51241) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
✓ Clip extracted


In [11]:
# ── Cell 10: Stage 2 — Extract Audio ─────────────────────────────────────────
import os
if "CLIP" not in globals(): CLIP = "workspace/clip.mp4"
AUDIO_RAW = "workspace/clip_audio.wav"
AUDIO_16K = "workspace/clip_audio_16k.wav"
AUDIO_REF = "workspace/clip_audio_ref44k.wav"

print("Extracting audio tracks...")
# 1. Raw audio
get_ipython().system(f"ffmpeg -y -i {CLIP} -vn -acodec pcm_s16le -ar 44100 {AUDIO_RAW} -loglevel warning")
# 2. 16k for Whisper
get_ipython().system(f"ffmpeg -y -i {AUDIO_RAW} -ac 1 -ar 16000 {AUDIO_16K} -loglevel warning")
# 3. 44k Reference for XTTS
get_ipython().system(f"ffmpeg -y -i {AUDIO_RAW} -ac 1 -ar 44100 {AUDIO_REF} -loglevel warning")
print("✓ Audio extracted (Raw, 16k, 44k Ref)")


Extracting audio tracks...
zsh(51262) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51262) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(51262) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(51263) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51263) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(51263) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
[aist#0:0/pcm_s16le @ 0x93ec18180] 

In [12]:
# ── Cell 11: Stage 3 — Transcribe with Whisper ───────────────────────────────
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import whisper

print('Loading Whisper medium model (downloading ~1.5 GB on first run)...')
model = whisper.load_model('medium')  # 'small' is faster; 'large' is more accurate

result = model.transcribe(AUDIO_16K, language='en', word_timestamps=True, verbose=False)
ENGLISH_TEXT = result['text'].strip()

with open('workspace/transcript_en.txt', 'w') as f:
    f.write(ENGLISH_TEXT)

print(f'✓ Transcript:\n{ENGLISH_TEXT}')

Loading Whisper medium model (downloading ~1.5 GB on first run)...


/Users/vikashkumar/Desktop/Supernan/venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 2001/2001 [00:22<00:00, 89.98frames/s]

✓ Transcript:
Clean your body, take bath, put deodorant, put a cloth Why do you do all these? Because you are always around your child Your child is always around you So if you smell your child's body or smell their breath Children will not like it They will not come to you


In [13]:
# ── Cell 12: Stage 4 — Translate to Hindi (IndicTrans2) ──────────────────────
import torch, os
if "DEVICE" not in globals(): DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if "ENGLISH_TEXT" not in globals():
    if os.path.exists("workspace/transcript_en.txt"):
        with open("workspace/transcript_en.txt", "r") as f: ENGLISH_TEXT = f.read().strip()
        print("✓ ENGLISH_TEXT recovered from disk")
    else: raise NameError("ENGLISH_TEXT missing. Run Cell 11.")

def translate_indictrans2(text):
    try:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        from IndicTransTokenizer import IndicProcessor
        MODEL = "ai4bharat/indictrans2-en-indic-1B"
        print("Loading IndicTrans2...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, trust_remote_code=True).to(DEVICE)
        ip = IndicProcessor(inference=True)
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        batch = ip.preprocess_batch(sentences, src_lang="eng_Latn", tgt_lang="hin_Deva")
        inputs = tokenizer(batch, truncation=True, padding="longest", return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, num_beams=5, max_length=256)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        postprocessed = ip.postprocess_batch(decoded, lang="hin_Deva")
        return " ".join(postprocessed)
    except Exception as e:
        print(f"IndicTrans2 failed: {e}")
        return None

print("Attempting Hindi translation...")
HINDI_TEXT = translate_indictrans2(ENGLISH_TEXT)

if not HINDI_TEXT:
    print("⚠️ IndicTrans2 failed. Attempting Google Translate fallback...")
    try:
        from deep_translator import GoogleTranslator
        HINDI_TEXT = GoogleTranslator(source="en", target="hi").translate(ENGLISH_TEXT)
        print("✓ Google Translate success!")
    except Exception as ge:
        print(f"❌ Google Translate failed: {ge}. Using English as last resort.")
        HINDI_TEXT = ENGLISH_TEXT

with open("workspace/translation_hi.txt", "w", encoding="utf-8") as f: f.write(HINDI_TEXT)
print(f"✓ Hindi translation completed.")
print(f"Text: {HINDI_TEXT[:100]}...")


Attempting Hindi translation...
IndicTrans2 failed: No module named 'IndicTransTokenizer'
⚠️ IndicTrans2 failed. Attempting Google Translate fallback...
✓ Google Translate success!
✓ Hindi translation completed.
Text: अपना शरीर साफ़ करो, नहाओ, डियोडरेंट लगाओ, कपड़ा डालो तुम ये सब क्यों करते हो? क्योंकि आप हमेशा अपने ...


In [14]:
# ── Cell 13: Stage 5 — Coqui XTTS v2 Voice Cloning ──────────────────────────
import os, re, torch
from TTS.api import TTS

# Recovery & Guards
if "DEVICE" not in globals(): DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if "AUDIO_REF" not in globals(): AUDIO_REF = "workspace/clip_audio_ref44k.wav"
if "HINDI_TEXT" not in globals() or not HINDI_TEXT.strip():
    if os.path.exists("workspace/translation_hi.txt"):
        with open("workspace/translation_hi.txt", "r", encoding="utf-8") as f: HINDI_TEXT = f.read().strip()
    if not globals().get("HINDI_TEXT"):
        if "ENGLISH_TEXT" in globals(): HINDI_TEXT = ENGLISH_TEXT
        else: raise ValueError("HINDI_TEXT empty and no fallback found.")

# 🛡️ PyTorch 2.6+ Unpickling Fix (Comprehensive)
try:
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts.xtts_audio_config import XttsAudioConfig
    import torch.serialization
    torch.serialization.add_safe_globals([XttsConfig, XttsAudioConfig])
    print("✓ Added XTTS configs to PyTorch safe globals")
except Exception as e:
    print(f"⚠️ Could not add safe globals: {e}")

# Final fallback: Monkeypatch torch.load to allow non-weights_only loading
if not hasattr(torch.load, "__supernan_patch__"):
    orig_load = torch.load
    def patched_load(*args, **kwargs):
        if "weights_only" not in kwargs: kwargs["weights_only"] = False
        return orig_load(*args, **kwargs)
    patched_load.__supernan_patch__ = True
    torch.load = patched_load
    print("✓ Applied torch.load security bypass monkeypatch")

os.environ["COQUI_TOS_AGREED"] = "1"
TTS_RAW = "workspace/tts_raw.wav"

print("Loading Coqui XTTS v2...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)

print(f"Synthesizing Hindi audio...")
tts.tts_to_file(
    text=HINDI_TEXT,
    speaker_wav=AUDIO_REF,
    language="hi",
    file_path=TTS_RAW,
    temperature=0.7,
    length_penalty=1.0,
    repetition_penalty=2.0
)
print(f"✓ Synthesized: {TTS_RAW}")


/Users/vikashkumar/Desktop/Supernan/venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


⚠️ Could not add safe globals: No module named 'TTS.tts.models.xtts.xtts_audio_config'; 'TTS.tts.models.xtts' is not a package
✓ Applied torch.load security bypass monkeypatch
Loading Coqui XTTS v2...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Synthesizing Hindi audio...
 > Text splitted to sentences.
['अपना शरीर साफ़ करो, नहाओ, डियोडरेंट लगाओ, कपड़ा डालो तुम ये सब क्यों करते हो?', 'क्योंकि आप हमेशा अपने बच्चे के आसपास रहते हैं आपका बच्चा हमेशा आपके आसपास रहता है इसलिए अगर आप अपने बच्चे के शरीर को सूंघेंगे या उनकी सांसों को सूंघेंगे तो बच्चों को यह पसंद नहीं आएगा वे आपके पास नहीं आएंगे']


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 > Processing time: 20.826693058013916
 > Real-time factor: 1.0180104408943544
✓ Synthesized: workspace/tts_raw.wav


In [15]:
# ── Cell 14: Stage 6 — Adjust Audio Speed ────────────────────────────────────
import subprocess, os
def get_duration(path):
    if not os.path.exists(path): return 0.0
    r = subprocess.run(["ffprobe","-v","quiet","-show_entries","format=duration","-of","csv=p=0",path],
                       capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0.0

if "CLIP" not in globals(): CLIP = "workspace/clip.mp4"
if "TTS_RAW" not in globals(): TTS_RAW = "workspace/tts_raw.wav"
TTS_ADJ = "workspace/tts_adjusted.wav"

def build_atempo(ratio):
    filters = []
    r = ratio
    while r < 0.5: filters.append("atempo=0.5"); r /= 0.5
    while r > 2.0: filters.append("atempo=2.0"); r /= 2.0
    filters.append(f"atempo={r:.6f}")
    return ",".join(filters)

tts_dur = get_duration(TTS_RAW)
clip_dur = get_duration(CLIP)

if tts_dur > 0 and clip_dur > 0:
    ratio = tts_dur / clip_dur
    atempo = build_atempo(ratio)
    print(f"Adjusting speed (ratio: {ratio:.3f})...")
    # 1. Apply atempo. 2. Pad with silence (apad) to ensure it reaches clip_dur. 3. Crop (-t) to exact clip_dur.
    get_ipython().system(f"ffmpeg -y -i {TTS_RAW} -filter:a '{atempo},apad' -t {clip_dur} -ar 44100 {TTS_ADJ} -loglevel warning")
    print(f"✓ Audio adjusted and strictly padded to {clip_dur:.2f}s for perfect sync")
else: print("⚠️ Missing files, skipping speed adjustment")


Adjusting speed (ratio: 0.940)...
zsh(51426) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51426) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(51426) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
[aist#0:0/pcm_s16le @ 0xaaac18180] Guessed Channel Layout: mono
✓ Audio adjusted to 20.00s


In [16]:
# ── Cell 15: Stage 7 — VideoReTalking Lip Sync ───────────────────────────────
import sys, os, subprocess
if "CLIP" not in globals(): CLIP = "workspace/clip.mp4"
if "TTS_ADJ" not in globals(): TTS_ADJ = "workspace/tts_adjusted.wav"
LIPSYNC = "workspace/lipsync.mp4"

# Path fixes for VRT
for p in ["VideoReTalking", "VideoReTalking/third_part", "VideoReTalking/third_part/face3d"]:
    path = os.path.abspath(p)
    if path not in sys.path: sys.path.append(path)

print("Running VideoReTalking (Stage 7)...")
res = subprocess.run([sys.executable, "VideoReTalking/inference.py", "--face", CLIP, "--audio", TTS_ADJ, "--outfile", LIPSYNC, "--checkpoint_dir", "models/VideoReTalking"])

if res.returncode != 0:
    print("⚠️ VRT failed, falling back to simple mux.")
    get_ipython().system(f"ffmpeg -y -i {CLIP} -i {TTS_ADJ} -map 0:v -map 1:a -c:v copy -shortest {LIPSYNC} -loglevel warning")

from IPython.display import Video
Video(LIPSYNC, width=640)


Running VideoReTalking (Stage 7)...


Traceback (most recent call last):
  File "/Users/vikashkumar/Desktop/Supernan/VideoReTalking/inference.py", line 12, in <module>
    from third_part.face3d.util.preprocess import align_img
  File "/Users/vikashkumar/Desktop/Supernan/VideoReTalking/third_part/face3d/util/__init__.py", line 2, in <module>
    from face3d.util import *
ModuleNotFoundError: No module named 'face3d'


⚠️ VRT failed, falling back to simple mux.
zsh(51436) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51436) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(51436) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
[aist#1:0/pcm_s16le @ 0xaaec18900] Guessed Channel Layout: mono


In [ ]:
# ── Cell 16: Stage 8 — GFPGAN Face Enhancement ───────────────────────────────
SKIP_ENHANCEMENT = False  # Set to True to skip this slow stage and get output immediately
FINAL = "output/final_dubbed.mp4"
FRAMES_DIR = "workspace/frames_raw"
ENHANCED_DIR = "workspace/gfpgan_out"

import os, subprocess, torch

if SKIP_ENHANCEMENT:
    print("⏩ SKIP_ENHANCEMENT is True. Copying lip-sync video to final output...")
    get_ipython().system(f"cp {LIPSYNC} {FINAL}")
    print(f"✓ Final output (no enhancement): {FINAL}")
else:
    os.makedirs(FRAMES_DIR, exist_ok=True)
    print("Extracting frames...")
    get_ipython().system(f"ffmpeg -y -i {LIPSYNC} -qscale:v 1 -qmin 1 {FRAMES_DIR}/frame_%06d.png -loglevel warning")

    print("Running GFPGAN face restoration...")
    if torch.cuda.is_available():
        PROCESSOR = "cuda"
    elif torch.backends.mps.is_available():
        PROCESSOR = "mps"
    else:
        PROCESSOR = "cpu"
    print(f"Device detected: {PROCESSOR}")
    get_ipython().system(f"python GFPGAN/inference_gfpgan.py --ext png --device {PROCESSOR} -i {FRAMES_DIR} -o {ENHANCED_DIR} -v 1.4 -s 1 --bg_upsampler None")

    fps_r = subprocess.run(["ffprobe","-v","quiet","-select_streams","v:0",
                            "-show_entries","stream=r_frame_rate","-of","csv=p=0", LIPSYNC],
                           capture_output=True, text=True)
    FPS = fps_r.stdout.strip()

    RESTORED = f"{ENHANCED_DIR}/restored_imgs"
    print("Re-encoding final video...")
    get_ipython().system(f"ffmpeg -y -framerate {FPS} -pattern_type glob -i '{RESTORED}/*.png' -i {LIPSYNC} -map 0:v -map 1:a -c:v libx264 -crf 17 -preset slow -c:a aac -shortest {FINAL} -loglevel warning")

    final_dur = get_duration(FINAL)
    print(f"✓ Final output: {FINAL} ({final_dur:.2f}s)")

from IPython.display import Video
Video(FINAL, width=640)


Extracting frames...


zsh(51437) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(51437) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(51437) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
[swscaler @ 0x996618000] No accelerated colorspace conversion found from yuv420p to rgb24.
[swscaler @ 0x996618000] [swscaler @ 0x996628000] No accelerated colorspace conversion found from yuv420p to rgb24.
[swscaler @ 0x996618000] [swscaler @ 0x996638000] No accelerated colorspace conversion found from yuv420p to rgb24.
[swscaler @ 0x996618000] [swscaler @ 0x996648000] No accelerated colorspace conversion found from yuv420p to rgb24.
[swscaler @ 0x996618000] [swscaler @ 0x996658000] No accelerated colorspace conversion found

In [ ]:
# ── Cell 17: Download Output ──────────────────────────────────────────────────
import os
try:
    from google.colab import files
    files.download(FINAL)
    print(f"✓ Download started for {FINAL}")
except ImportError:
    print(f"✓ Output saved to: {os.path.abspath(FINAL)}")
    print("   (Local machine detected, skipping browser download)")


---
## Pipeline Summary

| Stage | Tool | Duration |
|---|---|---|
| Clip Extract | ffmpeg | ~5s |
| Audio Extract | ffmpeg | ~2s |
| Transcription | Whisper medium | ~30s |
| Translation | IndicTrans2 | ~60s |
| Voice Clone | Coqui XTTS v2 | ~3-5 min |
| Speed Adjust | ffmpeg atempo | ~5s |
| Lip Sync | VideoReTalking | ~10-20 min |
| Face Enhance | GFPGAN v1.4 | ~2-3 min |

**Total on free Colab T4: ~15-30 minutes for a 15-second clip**

**Cost: ₹0**